In [6]:
"""
Script 01 — Wasserstein W1 sur données agrégées (tous SNRs)
============================================================

Tâche unique : calculer la distance de Wasserstein W1 entre les distributions
réelles (agrégées sur les 6 SNRs) et les distributions simulées par chaque modèle
(Linear, NonLinear1, GainModulation), en 5-fold cross-validation.

Source des données : Distributions_Active_late_full.csv
    - Modèles utilisés : real_flat (snr=all) et model_flat (snr=all)
    - 20 participants, 5 folds, 3 modèles comparés

Sorties :
    - Affichage console : tableau récapitulatif + test Wilcoxon
    - Figure : boxplot du W1 moyen par modèle (population + individus)
"""

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, wilcoxon
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Headless mode
import os

# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────

NOTEBOOK_DIR = '/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDMODEL/Data_SoundGOOD/LEAD_ExperimentalFolder/SOUNDMODEL_clean'
DATA_PATH = os.path.join(NOTEBOOK_DIR, '../Distributions_Active_late.csv')
OUTPUT_DIR = NOTEBOOK_DIR

MODELS = {
    'linear': 'Linear',
    'nonlinear': 'NonLinear1',
    'gainmodul': 'GainModulation',
}
N_PARTICIPANTS = 20
N_FOLDS = 5


# ─────────────────────────────────────────────────────────────────────────────
# Chargement des données
# ─────────────────────────────────────────────────────────────────────────────

print("Chargement du CSV...")
df = pd.read_csv(DATA_PATH)
data_cols = [c for c in df.columns if c.startswith('idx_')]
print(f"  Shape: {df.shape} | Modèles disponibles: {sorted(df['model'].unique())}")

# ─────────────────────────────────────────────────────────────────────────────
# Agrégation des SNRs : créer des lignes avec snr='all'
# ─────────────────────────────────────────────────────────────────────────────
print("\nAgrégation des SNRs pour créer snr='all'...")
aggregated_distributions = {}
for participant in df['participant'].unique():
    for fold in df['fold'].unique():
        for model in df['model'].unique():
            subset = df[
                (df['participant'] == participant) &
                (df['fold'] == fold) &
                (df['model'] == model)
            ]
            if len(subset) > 0:
                key = (participant, fold, model)
                all_vals = []
                for _, row in subset.iterrows():
                    for col in data_cols:
                        val = pd.to_numeric(row[col], errors='coerce')
                        if not pd.isna(val):
                            all_vals.append(val)
                aggregated_distributions[key] = np.array(all_vals)
print(f"  Agrégation complétée : {len(aggregated_distributions)} combinaisons créées")


def get_distribution(participant, fold, model):
    """Retourne les valeurs agrégées (tous SNRs) non-NaN."""
    key = (participant, fold, model)
    if key not in aggregated_distributions:
        raise ValueError(f"Aucune donnée pour participant={participant}, fold={fold}, model={model}")
    return aggregated_distributions[key]


# ─────────────────────────────────────────────────────────────────────────────
# Calcul W1 par participant et par fold
# ─────────────────────────────────────────────────────────────────────────────

print("\nCalcul W1 pour chaque participant / fold / modèle...")
results = {label: np.zeros((N_PARTICIPANTS, N_FOLDS)) for label in MODELS.values()}

for p in range(N_PARTICIPANTS):
    for f in range(N_FOLDS):
        real = get_distribution(p, f, 'real')
        for model_key, model_label in MODELS.items():
            sim = get_distribution(p, f, model_key)
            results[model_label][p, f] = wasserstein_distance(real, sim)

# ─────────────────────────────────────────────────────────────────────────────
# Agrégation : score par participant = moyenne des 5 folds
# ─────────────────────────────────────────────────────────────────────────────

scores = {label: results[label].mean(axis=1) for label in MODELS.values()}

print("\n=== Résultats W1 agrégé (moyenne sur 5 folds) ===")
print(f"{'Modèle':<20} {'Mean':>8} {'Median':>8} {'Std':>8}")
print("-" * 46)
for label in MODELS.values():
    s = scores[label]
    print(f"{label:<20} {s.mean():>8.4f} {np.median(s):>8.4f} {s.std():>8.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# Tests statistiques : Wilcoxon signé (comparaisons par paires)
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Tests de Wilcoxon (comparaisons par paires) ===")
model_labels = list(MODELS.values())
for i in range(len(model_labels)):
    for j in range(i + 1, len(model_labels)):
        a, b = model_labels[i], model_labels[j]
        stat, p = wilcoxon(scores[a], scores[b])
        better = a if scores[a].mean() < scores[b].mean() else b
        print(f"  {a} vs {b}: p={p:.4f} — gagne: {better}")

# ─────────────────────────────────────────────────────────────────────────────
# Modèle gagnant par participant
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Modèle gagnant par participant (W1 le plus bas) ===")
winner_counts = {label: 0 for label in MODELS.values()}
for p in range(N_PARTICIPANTS):
    winner = min(MODELS.values(), key=lambda m: scores[m][p])
    winner_counts[winner] += 1
    print(f"  P{p:02d}: {winner:<20} (W1={scores[winner][p]:.4f})")

print(f"\nRécapitulatif : {winner_counts}")

# ─────────────────────────────────────────────────────────────────────────────
# Figure
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
x_labels = list(MODELS.values())
data_to_plot = [scores[label] for label in x_labels]

bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)
colors = ['#4C72B0', '#DD8452', '#55A868']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Overlay individual participants
for i, label in enumerate(x_labels):
    jitter = np.random.uniform(-0.1, 0.1, N_PARTICIPANTS)
    ax.scatter(np.full(N_PARTICIPANTS, i + 1) + jitter, scores[label],
               color='k', alpha=0.5, s=20, zorder=5)

ax.set_ylabel('W1 moyen (agrégé)', fontsize=12)
ax.set_title('Wasserstein W1 — Données agrégées (tous SNRs)', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'ModelComparison_Wasserstein_result.png')
plt.savefig(fig_path, dpi=150)
plt.close()

Chargement du CSV...
  Shape: (2400, 5004) | Modèles disponibles: ['gainmodul', 'linear', 'nonlinear', 'real']

Agrégation des SNRs pour créer snr='all'...
  Agrégation complétée : 400 combinaisons créées

Calcul W1 pour chaque participant / fold / modèle...

=== Résultats W1 agrégé (moyenne sur 5 folds) ===
Modèle                   Mean   Median      Std
----------------------------------------------
Linear                 0.1871   0.1900   0.0312
NonLinear1             0.1637   0.1684   0.0259
GainModulation         0.1279   0.1276   0.0221

=== Tests de Wilcoxon (comparaisons par paires) ===
  Linear vs NonLinear1: p=0.0000 — gagne: NonLinear1
  Linear vs GainModulation: p=0.0000 — gagne: GainModulation
  NonLinear1 vs GainModulation: p=0.0000 — gagne: GainModulation

=== Modèle gagnant par participant (W1 le plus bas) ===
  P00: GainModulation       (W1=0.1597)
  P01: GainModulation       (W1=0.1480)
  P02: GainModulation       (W1=0.1341)
  P03: GainModulation       (W1=0.1521)
  

/tmp/ipykernel_830496/2288894842.py:146: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)
